# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [9]:
import pandas as pd
import numpy as np

# Create a sample dataset
np.random.seed(42)

n = 200

df_sample = pd.DataFrame({
    "content_id": range(1000, 1000 + n),
    "impressions": np.random.randint(100, 10000, n),
    "clicks": np.random.randint(10, 1000, n),
    "average_position": np.random.uniform(1, 30, n),
    "bounce_rate": np.random.uniform(0.2, 0.9, n),
    "time_on_page": np.random.uniform(30, 300, n),
    "word_count": np.random.randint(300, 2000, n),
    "content_type": np.random.choice(
        ["article", "blog_post", "product_page"], n
    ),
    "last_modified_days": np.random.randint(1, 365, n),
    "refresh_opportunity_score": np.random.uniform(0, 1, n) # The target label
})

# Feature Engineering
df_sample["ctr"] = df_sample["clicks"] / df_sample["impressions"]

# Handle missing values
df_sample["bounce_rate"] = df_sample["bounce_rate"].fillna(df_sample["bounce_rate"].median())

# One-hot encode categorical feature
df_sample = pd.get_dummies(df_sample, columns=["content_type"], drop_first=True)

# Define features (X) by excluding the label and content_id
X = df_sample.drop(columns=["refresh_opportunity_score", "content_id"])

print("Feature Vector Shape:", X.shape)
display(df_sample.head())
display(X.head())

Feature Vector Shape: (200, 10)


,content_id,impressions,clicks,average_position,bounce_rate,time_on_page,word_count,last_modified_days,refresh_opportunity_score,ctr,content_type_blog_post,content_type_product_page
0,1000,7370,991,4.962019,0.258024,161.708646,1967,269,0.247540,0.134464,False,True
1,1001,960,916,21.558419,0.569859,259.720730,1587,120,0.558413,0.954167,True,False
2,1002,5490,520,17.031779,0.505435,53.729659,1224,89,0.225540,0.094718,False,False
3,1003,5291,761,9.598794,0.761476,247.583519,354,329,0.160825,0.143829,False,True
4,1004,5834,153,13.173645,0.884530,45.026442,1966,68,0.651519,0.026226,False,True


,impressions,clicks,average_position,bounce_rate,time_on_page,word_count,last_modified_days,ctr,content_type_blog_post,content_type_product_page
0,7370,991,4.962019,0.258024,161.708646,1967,269,0.134464,False,True
1,960,916,21.558419,0.569859,259.720730,1587,120,0.954167,True,False
2,5490,520,17.031779,0.505435,53.729659,1224,89,0.094718,False,False
3,5291,761,9.598794,0.761476,247.583519,354,329,0.143829,False,True
4,5834,153,13.173645,0.884530,45.026442,1966,68,0.026226,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Feature Notes

| Feature | Meaning | Missing Values Handling | Categorical | Available Before Prediction |
|---|---|---|---|---|
| `content_id` | Unique identifier for each piece of content | N/A (excluded) | No | Yes |
| `impressions` | Number of times the content was shown | None | No | Yes |
| `clicks` | Number of times the content was clicked | None | No | Yes |
| `ctr` | Click-Through Rate (`clicks` / `impressions`) | Calculated from other features | No | Yes |
| `average_position` | Average search engine ranking position | None | No | Yes |
| `bounce_rate` | Percentage of users who leave the page after viewing only one page | Filled with median | No | Yes |
| `time_on_page` | Average time users spend on the content page | None | No | Yes |
| `word_count` | Number of words in the content | None | No | Yes |
| `last_modified_days` | Number of days since the content was last updated | None | No | Yes |
| `content_type_blog_post` | Indicates if content is a blog post (after one-hot encoding) | N/A | Yes | Yes |
| `content_type_product_page` | Indicates if content is a product page (after one-hot encoding) | N/A | Yes | Yes |
| `refresh_opportunity_score` | Target variable: score indicating refresh opportunity | N/A (excluded as label) | No | **No** (this is the label we predict) |

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [10]:
# Check for potential leakage columns

possible_leakage = [
    "refresh_opportunity_score",
    "future_clicks",
    "future_impressions",
    "label",
    "target"
]

print("Leakage Check:\n")

for col in possible_leakage:
    if col in df_sample.columns:
        print(f"⚠️ {col} -> Possible data leakage. Exclude from training.")
    else:
        print(f"✅ {col} -> Not found.")

Leakage Check:

⚠️ refresh_opportunity_score -> Possible data leakage. Exclude from training.
✅ future_clicks -> Not found.
✅ future_impressions -> Not found.
✅ label -> Not found.
✅ target -> Not found.


## Leakage Findings

No future information was used as a feature.

The target column (`refresh_opportunity_score`) is excluded from the feature vector.

No future clicks, future impressions, or label-derived columns were found.

Only information available before prediction is used.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## What I Excluded and Why

| Excluded Field | Reason |
|----------------|--------|
| refresh_opportunity_score | This is the target (label) and must not be used as a feature because it would cause data leakage. |
| content_id | Identifier only; it has no predictive value. |
| future_clicks | Future information is not available at prediction time and would cause leakage. |
| future_impressions | Future information is not available at prediction time and would cause leakage. |
| label / target | These directly represent the prediction outcome and cannot be used as input features. |
| Personally Identifiable Information (PII) | Excluded for privacy and because it is not relevant to prediction. |